In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('Dataset.csv')

In [9]:
for col in ['director', 'country', 'rating', 'listed_in', 'type']:
    df[col] = df[col].fillna('')

df['content_soup'] = (
    df['listed_in'] + ' ' +
    df['director'] + ' ' +
    df['country'] + ' ' +
    df['type'] + ' ' +
    df['rating']
)

df[['title', 'content_soup']].head()

,title,content_soup
0,Dick Johnson Is Dead,Documentaries Kirsten Johnson United States Mo...
1,Ganglands,"Crime TV Shows, International TV Shows, TV Act..."
2,Midnight Mass,"TV Dramas, TV Horror, TV Mysteries Mike Flanag..."
3,Confessions of an Invisible Girl,"Children & Family Movies, Comedies Bruno Garot..."
4,Sankofa,"Dramas, Independent Movies, International Movi..."


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['content_soup'])

print(tfidf_matrix.shape)

(8790, 6569)


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(cosine_sim.shape)

(8790, 8790)


In [14]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def recommend(title, num_recommendations=5):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:num_recommendations+1]
    title_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[title_indices]

print(recommend('Ganglands'))

1183         Sentinelle
2142    Earth and Blood
6738              Lupin
8408         Crime Time
6698             Mortel
Name: title, dtype: object


In [16]:
def evaluate_recommendations(title, num_recommendations=5):
    recommended_titles = recommend(title, num_recommendations)
    input_genres = set(df.loc[df['title'] == title, 'listed_in'].values[0].split(', '))

    overlap_scores = []
    for rec_title in recommended_titles:
        rec_genres = set(df.loc[df['title'] == rec_title, 'listed_in'].values[0].split(', '))
        overlap = len(input_genres & rec_genres) / len(input_genres)
        overlap_scores.append(overlap)

    avg_overlap = sum(overlap_scores) / len(overlap_scores)
    print(f"Input title: {title}")
    print(f"Genres: {input_genres}")
    print(f"Average genre overlap with recommendations: {avg_overlap:.2f}")
    return avg_overlap

evaluate_recommendations('Mariposa')

Input title: Mariposa
Genres: {'Comedies', 'Romantic Movies', 'International Movies'}
Average genre overlap with recommendations: 0.40


0.4